## 1. 今日の量子コンピュータの問題

- Noisy Intermediate-Scale Quantum (NISQ) デバイス
    - 量子回路が深くなる（ゲート数が多くなる）ほど、誤差が大きくなる
    - 十分な量子ビット数ではない
- 量子デバイスは特別なゲート演算のみが用意されている
- 特定のqubits間の量子ビット演算(multi qubit operation)しか用意されていない
- それぞれの量子デバイスに対して、量子ソフトウェアツールキットが用意されてる


### 1-1. TKETとは
- Quantum Software Development Kit
- TKETに実装されている回路最適化はC++で実装
- pythonモジュール　`pytket`で利用可能
- 最適化コンパイラ：　ユーザーフレンドリーな回路→量子デバイスで実行可能な回路に変換可能
    - Language-agnostic (多くの量子プログラミングフレームワーク(qiskit, Cirq, etc)をサポート)
    - Retagetable (多くの量子デバイス(IBM, Quantinuum, Amazon Braket(IonQ, Rigetti, IQM, etc)をサポート)
    - Circuit Optimisation (量子計算時に生じるデバイスエラーの影響を最小化。デバイス依存＆デバイス非依存のものが実装)
    
<img src="./fig/tket1_revised.png" width="750">



#### 参照
- [pytket ドキュメント](https://docs.quantinuum.com/tket/api-docs/)
- [pytket ユーザーガイド](https://docs.quantinuum.com/tket/user-guide/)
- [t|ket⟩ : A Retargetable Compiler for NISQ Devices](https://arxiv.org/abs/2003.10611)
- [TKET slack channel](https://join.slack.com/t/tketusers/shared_invite/zt-2aoan2s87-WDdZQeY2dbJQgAQE6O~3qg)

<img src="./fig/slack-qr.png" width="250">


### 1-2. pytketと拡張 pytket (python パッケージ)
Python 3.11,12で動作確認をしています。

|  パッケージ |  概要  |
| :---- | :---- |
|  pytket  |  TKETを利用するためのpython モジュール  ( available for python3.10 or higher )|
|  pytket-quantinuum  |  Quantinuumエミュレータを利用するためのpytket-extension  |
|  pytket-quantinuum[pecos]  |  Quantinuumエミュレータのパッケージ  |
|  pytket-qulacs  |  Qulacsシミュレータを利用するためのpytket-extension  |

<img src="./fig/tket2.png" width="850">

In [ ]:
!pip install pytket-quantinuum[pecos]
# こちらの作業は一度でよい。インストール後、Kernelをリスタートする必要がある

## 2. 量子回路を作成する
ここでは `TKET`でGreenberger–Horne–Zeilinger状態を作成する。

### 2-1. `TKET`で4 qubitsのGreenberger–Horne–Zeilinger状態(GHZ状態)を作成
$$ |\Psi\rangle = \frac{1}{\sqrt{2}}(|0000\rangle+|1111\rangle)$$

In [4]:
from pytket import Circuit
from pytket.circuit.display import render_circuit_jupyter

ghz4 = Circuit(4)
ghz4.H(0).CX(0,1).CX(0,2).CX(0,3)
ghz4.measure_all()
render_circuit_jupyter(ghz4)

#### 研究者向け：量子回路のLatexソースを生成することが可能

In [5]:
ghz4.to_latex_file("ghz.tex")

### 2-2. `pytket-quantinuum`でTKET 量子回路をQuantinuum エミュレータで計算

In [6]:
from pytket.extensions.quantinuum import QuantinuumBackend
quantinuum_backend = QuantinuumBackend(device_name ='H2-1LE')
quantinuum_ghz4 = quantinuum_backend.get_compiled_circuit(ghz4)
render_circuit_jupyter(quantinuum_ghz4)
render_circuit_jupyter(ghz4)

In [7]:
handle = quantinuum_backend.process_circuit(quantinuum_ghz4, n_shots=1000)
result = quantinuum_backend.get_result(handle)
counts = result.get_counts()
counts

Counter({(1, 1, 1, 1): 516, (0, 0, 0, 0): 484})

## 3. 量子回路の最適化
例えば、Hゲートを同じビットに連続して作用させるとゲート操作をしていないのと恒等な量子状態が得られます。

量子回路が深くなる（ゲート数が多くなる）ほど、誤差が大きくなるNISQ デバイスでは左辺のような状況は除去したい。

その他にも様々な恒等な関係があります。

In [8]:
from pytket.pauli import Pauli
from pytket.circuit import PauliExpBox, fresh_symbol, OpType
from pytket.passes import DecomposeBoxes
box = PauliExpBox([Pauli.I, Pauli.Z, Pauli.X, Pauli.Y], fresh_symbol('tm'))
from pytket.utils import Graph
import numpy as np

def get_random_pauli_gadgets(n_qubits: int, n_pauli_gadgets: int, max_entangle: int) -> Circuit:
    """ランダムにユニタリゲートを追加したパラメタ付き量子回路を準備する関数"""
    """ n_qubits個のqubitから成る回路に最大n_pauli_gadgets個のPauliExpBox(expの型に、ここでは長さmax_entangleのPauli stringが乗ったもの)を追加したものを準備し、そのPauliExpBoxを分解した回路を返している"""
    paulis = [Pauli.I, Pauli.X, Pauli.Y, Pauli.Z]
    circ = Circuit(n_qubits)
    for i in range(n_pauli_gadgets):
        ls_paulis = [np.random.choice(paulis) for k in range(max_entangle)]
        if ls_paulis.count(Pauli.Y) % 2 == 0:
            continue
        if len(ls_paulis) - ls_paulis.count(Pauli.I) <= 1:
            continue
        qubits = np.random.choice(
            [i for i in range(n_qubits)], size=max_entangle, replace=False
        )
        box = PauliExpBox(ls_paulis, fresh_symbol('a'))
        circ.add_pauliexpbox(box, sorted(qubits))
    DecomposeBoxes().apply(circ)
    return circ

### PauliSquash 関数を利用した、量子回路の最適化
TKETには量子回路を最適化する様々な機能が用意されている。
ここでは、PauliSquash 関数を利用した回路の最適化を行う。

（PauliSquash 関数：CX ゲートとTK1ゲートで表現された量子回路を出力）

ランダムな量子回路を作成し、回路の深さとCXの深さを数える。

In [9]:
circ = get_random_pauli_gadgets(
    n_qubits=8, n_pauli_gadgets=300, max_entangle=5
)
print('Circuit depth: ', circ.depth())
print('CX depth: ', circ.depth_by_type(OpType.CX))
#render_circuit_jupyter(circ)

Circuit depth:  927
CX depth:  568


PauliSquash 関数を使って、量子回路の最適化

In [10]:
# Circuit optimization by using compiler passes.
from pytket.passes import PauliSquash
circx = circ.copy()
PauliSquash().apply(circx)
print('Circuit depth: ', circx.depth())
print('CX depth: ', circx.depth_by_type(OpType.CX))
#render_circuit_jupyter(circx)

Circuit depth:  839
CX depth:  567
